# Encodec (HuggingFace) token/latent experiments

This notebook:
- loads HF `facebook/encodec_24khz`
- reads `.ecdc` (via your `utils.ecdc_utils.load_ecdc`)
- builds the per-level lookup table **by decoding each token index** (RNeNcodec-style)
- converts `(T, n_q)` token stacks to `(T,128)` latents (and per-level contributions)
- encodes audio to tokens
- verifies manual latent reconstruction matches HF `quantizer.decode` (with correct code layout)


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from transformers import EncodecModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "facebook/encodec_24khz"

model = EncodecModel.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

print("Loaded:", MODEL_ID)
print("Device:", DEVICE)
print("Model sampling rate:", getattr(model.config, "sampling_rate", None))
print("Codebook size (K):", getattr(model.config, "codebook_size", None))
print("Target bandwidths:", getattr(model.config, "target_bandwidths", None))


In [ ]:

from utils.ecdc_utils import load_ecdc  # confirmed OK in your setup
from utils.ecdc_utils import bandwidth_to_n_q, n_q_to_bandwidth

# Creates the lookup table for an Encodec model to use for token-> latent mapping
from utils.ecdc_utils import  build_E_eff_via_layer_decode

# Creates a "pool" of tokens that can be drawn on for style transfer purposes
from utils.ecdc_utils import  tokens_to_summary_latents

# returns tensor of latents for just one level of a token stack sequence
from utils.ecdc_utils import  token_level_to_latents


In [ ]:
# ---- 1) Load one .ecdc (tokens_TN is (T, n_q)) ----
dspath = "ecdc/ChirpPattern--irreg_exp-00.00--c-00--x-10.ecdc"
tokens_TN, scales, raw = load_ecdc(dspath)

print("tokens_TN:", tokens_TN.shape, tokens_TN.dtype)   # (T, n_q)
print("scales:", scales)
print("raw keys:", list(raw.keys()))
print("audio_length:", raw.get("audio_length"))


In [ ]:
# Create the Encodec code->latent lookup table ...... This is fast!
#__________________________________________________________________
K = int(getattr(model.config, "codebook_size", 1024))
n_q_data = int(tokens_TN.shape[1])
E_eff_QKD = build_E_eff_via_layer_decode(model, n_q=n_q_data, K=K, device=DEVICE)
print("E_eff_QKD:", E_eff_QKD.shape, E_eff_QKD.dtype, E_eff_QKD.device)

In [ ]:
# ---- 2a) Token stack (T,n_q) -> latent (T,128) via lookup+sum ----

z_T128 = tokens_to_summary_latents(tokens_TN, E_eff_QKD)
print("z_T128:", z_T128.shape, z_T128.dtype, z_T128.device)


In [ ]:
# ---- 2b) Individual level contribution (T,128) ----

# Example: level 0
z0 = token_level_to_latents(tokens_TN, 0, E_eff_QKD)
print("z0:", z0.shape)


In [ ]:
# ---- Check: sum of individual level contributions equals summary latent ----
@torch.no_grad()
def check_sum_property(tokens_TN: torch.Tensor, E_eff_QKD: torch.Tensor):
    z_full = tokens_to_summary_latents(tokens_TN, E_eff_QKD)
    z_sum = torch.zeros_like(z_full)
    for q in range(tokens_TN.shape[1]):
        z_sum.add_(token_level_to_latents(tokens_TN, q, E_eff_QKD))
    err = (z_full - z_sum).abs().max().item()
    print("max|z_full - sum(levels)| =", err)

check_sum_property(tokens_TN, E_eff_QKD)


In [ ]:
from utils.ecdc_utils import  ensure_BCT
# ---- 3) Encode audio -> codes, and verify manual latents match HF quantizer.decode ----
@torch.no_grad()
def verify_manual_matches_hf_quantizer_decode(model, audio_BCT: torch.Tensor, bandwidth=None, K: int = 1024):
    device = next(model.parameters()).device
    audio_BCT = ensure_BCT(audio_BCT).to(device)

    # Encode -> audio_codes
    enc = model.encode(audio_BCT) if bandwidth is None else model.encode(audio_BCT, bandwidth=bandwidth)
    codes = enc.audio_codes

    # Normalize to codes_BQT = (B, n_q, Tframes)
    if codes.ndim == 4:
        # [B, 1, n_q, T]
        codes_BQT = codes[:, 0]
    elif codes.ndim == 3:
        codes_BQT = codes
    else:
        raise ValueError(f"Unexpected audio_codes shape {tuple(codes.shape)}")

    B, n_q, Tframes = codes_BQT.shape

    # Build E_eff via layer.decode (exactly like RNeNcodec)
    E_eff_QKD = build_E_eff_via_layer_decode(model, n_q=n_q, K=K, device=device)
    D = E_eff_QKD.shape[-1]

    # Manual z_manual_BTD = (B, T, D)
    z_manual_BTD = torch.zeros(B, Tframes, D, device=device, dtype=E_eff_QKD.dtype)
    for q in range(n_q):
        idx = codes_BQT[:, q, :].reshape(-1)                       # (B*T,)
        e = F.embedding(idx, E_eff_QKD[q]).view(B, Tframes, D)     # (B,T,D)
        z_manual_BTD.add_(e)

    # HF quantizer.decode expects codes_QBT = (n_q, B, T)
    codes_QBT = codes_BQT.permute(1, 0, 2).contiguous()
    z_hf_BDT = model.quantizer.decode(codes_QBT)                   # (B,D,T)
    if z_hf_BDT.shape != (B, D, Tframes):
        raise RuntimeError(f"Unexpected decode output {tuple(z_hf_BDT.shape)} expected {(B,D,Tframes)}")

    z_hf_BTD = z_hf_BDT.permute(0, 2, 1).contiguous()              # (B,T,D)

    diff = (z_manual_BTD - z_hf_BTD).abs()
    print("codes_BQT:", tuple(codes_BQT.shape), "(B,n_q,Tframes)")
    print("z_manual:", tuple(z_manual_BTD.shape), "(B,T,D)")
    print("z_hf:    ", tuple(z_hf_BTD.shape), "(B,T,D)")
    print("max abs diff:", diff.max().item())
    print("mean abs diff:", diff.mean().item())
    print("bandwidth:", bandwidth, "| n_q:", n_q)

    return z_manual_BTD, z_hf_BTD, codes_BQT, enc, E_eff_QKD

# Quick smoke test with random audio (1 sec at 24kHz)
audio = torch.randn(1, 1, 24000) * 0.01
verify_manual_matches_hf_quantizer_decode(model, audio, bandwidth=None, K=K)


## Notes

- The critical correction is **HF expects `quantizer.decode(codes)` input shape `(n_q, B, T)`**, not `(B, n_q, T)`.
- The lookup table is built **by calling `layers[q].decode(idx_all)`**, matching your RNeNcodec approach.
- If you want your encode to produce a specific `n_q` (e.g. 8), call `model.encode(audio, bandwidth=6.0)`.


In [ ]:
# Example: force n_q = 8 by choosing bandwidth = 6.0 kbps (per your mapping)
audio = torch.randn(1, 1, 24000) * 0.01
bw = 6.0
z_manual, z_hf, codes_BQT, enc, E_eff = verify_manual_matches_hf_quantizer_decode(model, audio, bandwidth=bw, K=K)
print("n_q from encode:", codes_BQT.shape[1], "expected:", bandwidth_to_n_q(bw))
